# Public-source note

Paths have been sanitized. Notebook outputs are temporarily cleared in this draft while the workflow is being reorganized. For the final repository, the cleaned notebooks will be re-run and their output cells retained; any machine-specific paths printed in outputs will be removed or replaced with portable labels.


In [ ]:
suppressPackageStartupMessages({
  library(dplyr)
  library(ggplot2)
  library(data.table)
  library(GenomicRanges)
  library(BiocParallel)
  library(ComplexHeatmap)
  library(patchwork)
  library(reticulate)
  library(rtracklayer)
  library(tidyr)
    
library(circlize)
library(dplyr)
library(ggplot2)
library(data.table)
library(GenomicRanges)
library(BiocParallel)
library(ComplexHeatmap)
})

In [ ]:
figure_dir_old <- "/path/to/CEBP_analysis/data/ChromBPNet_Fig4/figures_atac_chromhmm/"

figure_dir <- "/path/to/CEBP_analysis/data/ChromBPNet_Fig4_V2/"
finmo_CWMs_dir <-"/path/to/CEBPB_seq/chrombpnet_hits/Combined_modisco_1000_5000000_all_CEBPB/pos_patterns/pos_patterns/merged/CWMs/"
tomtom_cwm_dir <- "/path/to/CEBPB_seq/chrombpnet_hits/Combined_modisco_1000_5000000_all_CEBPB/pos_patterns/pos_patterns/tomtom_cwm/"
modisco_html_path <- "/path/to/CEBPB_seq/chrombpnet_hits/Combined_modisco_1000_5000000_all_CEBPB/pos_patterns/pos_patterns/merged_modisco.html/motifs.html"

min_prop <- 0.05


In [ ]:
motifs_meme <- universalmotif::read_meme("/path/to/external_analysis/HumanReprogrammingMultiome/Iris/multiome_V2/tables/ChromBPNet_full//motifs.meme.txt")

In [ ]:
# Publication-facing motif annotation table.
# This compact TSV replaces the two historical working annotation files.
manual_ordered_joined_df <- read.delim(
  "fig5c_motif_annotations.tsv",
  header = TRUE,
  sep = "\t",
  stringsAsFactors = FALSE,
  check.names = FALSE
)

# Aliases retained so downstream cells from the original analysis remain readable.
kc_df <- manual_ordered_joined_df
manual_ordered_df <- manual_ordered_joined_df

colnames(manual_ordered_joined_df)


In [ ]:
head(kc_df)

In [ ]:
# Motif annotations are loaded from fig5c_motif_annotations.tsv above.


In [ ]:
# Keep the full publication-facing annotation table; no additional column reduction is required.


In [ ]:
# manual_ordered_joined_df is already loaded from fig5c_motif_annotations.tsv.


In [ ]:
head(manual_ordered_joined_df)

In [ ]:
nrow(manual_ordered_joined_df)

In [ ]:
# manual_ordered_df <- read.table(
#   paste("Finemo_PrATAC_patterns_0.05_tomtom_manual_match_manual_selected_with_KC_motifs_V4.csv",sep=""),
#   header = TRUE,
#   sep = ",",
#     # row.names = FALSE,
#   stringsAsFactors = FALSE
# )
# head(manual_ordered_df)

In [ ]:
length(unique(manual_ordered_joined_df$pattern))

In [ ]:
## keep only the rows where to_plot is 1
# manual_ordered_joined_df <- manual_ordered_joined_df[manual_ordered_df$to_plot == 1, ]

In [ ]:
length(unique(manual_ordered_joined_df$pattern))

In [ ]:
ordered_patterns <- manual_ordered_joined_df$pattern
ordered_matched_motif <- manual_ordered_joined_df$matched_motif
ordered_pattern_num <- manual_ordered_joined_df$pattern_num
ordered_pattern_Orientation <- manual_ordered_joined_df$Orientation
ordered_pattern_Selected_TF <- manual_ordered_joined_df$Selected_TF


In [ ]:
motif_list_modisco <- list()
motif_list_known  <- list()
ordered_matched_motif_updated <- character()   # empty vector

for (i in seq_along(ordered_patterns)) {
  pattern_sel   <- ordered_patterns[i]
  Orientation <- ordered_pattern_Orientation[i]
    Selected_TF <- ordered_pattern_Selected_TF[i]
    matched_motif <- ordered_matched_motif[i]
    

  motif_modisco_fc <- read.table(paste0(finmo_CWMs_dir, pattern_sel, "/modisco_fc.txt"))
  rownames(motif_modisco_fc) <- c("A","C","G","T")
  colnames(motif_modisco_fc) <- 1:ncol(motif_modisco_fc)
  motif_list_modisco[[paste("Modisco_FC", pattern_sel)]] <- as.matrix(motif_modisco_fc)

  tomtom <- read.table(paste0(tomtom_cwm_dir,pattern_sel,"_tomtom.csv"), header=T)


  PWM_motif_known <- universalmotif::filter_motifs(motifs_meme, name = matched_motif)[[1]]@motif
  colnames(PWM_motif_known) <- 1:ncol(PWM_motif_known)
  ori <- tomtom$Orientation[match(matched_motif, tomtom$Target_ID)]

  if (!is.na(ori) && ori == "-") {
      PWM_motif_known <- universalmotif::motif_rc(universalmotif::create_motif(PWM_motif_known))@motif
      colnames(PWM_motif_known) <- 1:ncol(PWM_motif_known)
    }
    
    ##store updated motif
  ordered_matched_motif_updated[i] <- matched_motif

  motif_list_known[[ paste(matched_motif, pattern_sel)]] <- as.matrix(PWM_motif_known)
}

In [ ]:
matched_motif

In [ ]:
tomtom_cwm_dir

In [ ]:
length(motif_list_known)

In [ ]:
length(motif_list_modisco)

In [ ]:

figure_file <-   "Motifs_logs_modisco_and_known_manual_order.pdf"

# helper once, outside the loop
logo_panel <- function(m, title) {
  if (is.null(m)) {
    ggplot() + theme_void() + ggtitle(paste(title, "(missing)"))
  } else {
    ggplot() +
      ggseqlogo::geom_logo(list(m), method = "custom") +
      ggseqlogo::theme_logo() +
      ggtitle(title) +
      theme(
        plot.title = element_text(size = 8, hjust = 0.5),
        axis.title.x = element_blank(),
        axis.title.y = element_blank(),
        axis.text.x  = element_blank(),
        axis.text.y  = element_blank(),
        axis.ticks.x = element_blank(),
        axis.ticks.y = element_blank(),
        panel.grid   = element_blank(),
        panel.border = element_blank(),
        plot.margin  = margin(0, 0, 0, 0)
      )
  }
}

get_tf_from_key <- function(k) {
  if (is.na(k) || length(k) == 0) return("missing")
  parts <- strsplit(k, " ")[[1]]
  if (length(parts) >= 3) parts[3] else "missing"
}

rows <- vector("list", length(ordered_patterns))

for (i in seq_along(ordered_patterns)) {
    pattern_sel <- ordered_patterns[i]
    matched_motif <- ordered_matched_motif_updated[i]
    
      key_fc     <- paste("Modisco_FC", pattern_sel)
    
    fc     <- motif_list_modisco[[key_fc]]
    known1 <-  motif_list_known[[paste(matched_motif, pattern_sel)]]
    
      pat_idx <- which(ordered_patterns == pattern_sel)
    
    
      p1 <- logo_panel(fc,    paste(ordered_pattern_Selected_TF[i],ordered_pattern_num[i]))
      p2 <- logo_panel(known1, matched_motif)
    
     rows[[i]] <- (p1 | p2) +
        plot_annotation(title = pattern_sel) &
        theme(
          plot.title = element_text(size = 9, hjust = 0),
          plot.margin = margin(0, 0, 0, 0)
        )
}

# stack all rows vertically into one figure
fig_all <- wrap_plots(rows, ncol = 1)

# set page height based on number of motifs
n <- length(rows)
# pdf(figure_file, width = 10, height = 80, useDingbats = FALSE)
pdf(figure_file, width = 10, height = 45, useDingbats = FALSE)
print(fig_all)
dev.off()



In [ ]:
library(ggplot2)
library(ggseqlogo)
library(patchwork)

figure_file <- "Motifs_logs_modisco_and_known_manual_order2.pdf"

## use the same order as manual_ordered_df
ordered_patterns <- manual_ordered_joined_df$pattern
pattern_clusters <- manual_ordered_joined_df$cluster

logo_panel <- function(m, title) {
  if (is.null(m)) {
    ggplot() +
      theme_void() +
      ggtitle(paste(title, "(missing)")) +
      theme(
        plot.title = element_text(size = 8, hjust = 0.5),
        plot.margin = margin(0, 0, 0, 0)
      )
  } else {
    ggplot() +
      ggseqlogo::geom_logo(list(m), method = "custom") +
      ggseqlogo::theme_logo() +
      ggtitle(title) +
      theme(
        plot.title = element_text(size = 8, hjust = 0.5),
        axis.title.x = element_blank(),
        axis.title.y = element_blank(),
        axis.text.x  = element_blank(),
        axis.text.y  = element_blank(),
        axis.ticks.x = element_blank(),
        axis.ticks.y = element_blank(),
        panel.grid   = element_blank(),
        panel.border = element_blank(),
        plot.margin  = margin(0, 0, 0, 0)
      )
  }
}

get_tf_from_key <- function(k) {
  if (is.na(k) || length(k) == 0) return("missing")
  parts <- strsplit(k, " ")[[1]]
  if (length(parts) >= 3) parts[3] else "missing"
}

rows <- vector("list", length(ordered_patterns))

for (i in seq_along(ordered_patterns)) {
  pattern_sel   <- ordered_patterns[i]
  matched_motif <- ordered_matched_motif_updated[i]

  key_fc <- paste("Modisco_FC", pattern_sel)

  fc     <- motif_list_modisco[[key_fc]]
  known1 <- motif_list_known[[paste(matched_motif, pattern_sel)]]

  p1 <- logo_panel(fc, paste(ordered_pattern_Selected_TF[i], ordered_pattern_num[i]))
  p2 <- logo_panel(known1, matched_motif)

  rows[[i]] <- (p1 | p2) +
    plot_annotation(title = pattern_sel) &
    theme(
      plot.title = element_text(size = 9, hjust = 0),
      plot.margin = margin(0, 0, 0, 0)
    )
}

## safety checks
if (length(pattern_clusters) != length(rows)) {
  stop("length(manual_ordered_joined_df$cluster) must match length(rows)")
}

## preserve cluster order as it appears in manual_ordered_df
pattern_clusters <- factor(pattern_clusters, levels = unique(pattern_clusters))

## split rows by cluster
rows_split <- split(rows, pattern_clusters)

## make one vertical block per cluster
cluster_blocks <- lapply(rows_split, function(one_cluster_rows) {
  wrap_plots(one_cluster_rows, ncol = 1)
})

## insert spacers between clusters to mimic ComplexHeatmap row gaps
fig_parts <- list()
heights_vec <- c()

for (i in seq_along(cluster_blocks)) {
  fig_parts[[length(fig_parts) + 1]] <- cluster_blocks[[i]]
  heights_vec <- c(heights_vec, 1)

  if (i < length(cluster_blocks)) {
    fig_parts[[length(fig_parts) + 1]] <- plot_spacer()
    heights_vec <- c(heights_vec, 0.06)
  }
}

## stack all cluster blocks vertically
fig_all <- wrap_plots(
  fig_parts,
  ncol = 1,
  heights = heights_vec
)

## scale page height to number of motif rows
n_rows <- length(rows)
n_clusters <- length(cluster_blocks)
pdf_height <- max(8, n_rows * 0.7 + (n_clusters - 1) * 0.35)

# pdf(figure_file, width = 10, height = pdf_height, useDingbats = FALSE)
pdf(figure_file, width = 10, height = 80, useDingbats = FALSE)
print(fig_all)
dev.off()

message("Saved: ", figure_file)

In [ ]:
length(motif_list_modisco)

In [ ]:
length(motif_list_known)